**Company House Table **creation****

In [0]:
from pyspark.sql.functions import current_timestamp, col

base_path = "/Volumes/company_risk_intelligence_platform/bronze/raw_data/companies_house/"

table_mapping = {
    "ch_filing_history": "filing_history",
    "ch_overview": "overview",
    "ch_people": "people"
}
#schema
catalog = "company_risk_intelligence_platform"
schema = "bronze"

def add_metadata_columns(df):
    return df.withColumn("last_update_ts", current_timestamp()) \
    .withColumn("file_path", col("_metadata.file_path"))

for table, folder in table_mapping.items():
    print(f"Processing {table}")

    input_path = base_path + folder + "/"

    df = spark.read \
        .option("multiline", "true") \
        .option("recursiveFileLookup", "true") \
        .json(input_path)
    df = add_metadata_columns(df)

    df.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(f"{catalog}.{schema}.{table}")

print("All tables processed")

In [0]:
tables_df = spark.sql("""
SHOW TABLES IN company_risk_intelligence_platform.bronze
""")

display(tables_df)

In [0]:
display(spark.sql("select * from company_risk_intelligence_platform.bronze.ch_overview"))

In [0]:
display(spark.sql("select * from company_risk_intelligence_platform.bronze.ch_people"))

In [0]:
display(spark.sql("select * from company_risk_intelligence_platform.bronze.ch_filing_history"))